# Explore CMS samples

First look at `data/samples/part_b_2023_sample.csv` and `part_d_2023_sample.csv` (5,000 rows each, pulled via `src/cms_outliers/data/pull_samples.py`). Goal: understand columns, cardinality, and data quality quirks before designing a schema.

## Load and check columns/dtypes

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

part_b = pd.read_csv("../data/samples/part_b_2023_sample.csv")
part_d = pd.read_csv("../data/samples/part_d_2023_sample.csv")

part_b.dtypes

Rndrng_NPI                         int64
Rndrng_Prvdr_Last_Org_Name           str
Rndrng_Prvdr_First_Name              str
Rndrng_Prvdr_MI                      str
Rndrng_Prvdr_Crdntls                 str
Rndrng_Prvdr_Ent_Cd                  str
Rndrng_Prvdr_St1                     str
Rndrng_Prvdr_St2                     str
Rndrng_Prvdr_City                    str
Rndrng_Prvdr_State_Abrvtn            str
Rndrng_Prvdr_State_FIPS              str
Rndrng_Prvdr_Zip5                  int64
Rndrng_Prvdr_RUCA                float64
Rndrng_Prvdr_RUCA_Desc               str
Rndrng_Prvdr_Cntry                   str
Rndrng_Prvdr_Type                    str
Rndrng_Prvdr_Mdcr_Prtcptg_Ind        str
HCPCS_Cd                             str
HCPCS_Desc                           str
HCPCS_Drug_Ind                       str
Place_Of_Srvc                        str
Tot_Benes                          int64
Tot_Srvcs                        float64
Tot_Bene_Day_Srvcs                 int64
Avg_Sbmtd_Chrg  

In [2]:
part_d.dtypes

Prscrbr_NPI                int64
Prscrbr_Last_Org_Name        str
Prscrbr_First_Name           str
Prscrbr_City                 str
Prscrbr_State_Abrvtn         str
Prscrbr_State_FIPS         int64
Prscrbr_Type                 str
Prscrbr_Type_Src             str
Brnd_Name                    str
Gnrc_Name                    str
Tot_Clms                   int64
Tot_30day_Fills          float64
Tot_Day_Suply              int64
Tot_Drug_Cst             float64
Tot_Benes                float64
GE65_Sprsn_Flag              str
GE65_Tot_Clms            float64
GE65_Tot_30day_Fills     float64
GE65_Tot_Drug_Cst        float64
GE65_Tot_Day_Suply       float64
GE65_Bene_Sprsn_Flag         str
GE65_Tot_Benes           float64
dtype: object

## Cardinality and sampling bias check

The API returns rows ordered by ascending NPI, and our sample is just the first 5,000 rows — so this is **not a random sample**, it's whichever providers happen to have the lowest NPIs. Worth confirming explicitly so we don't draw population-level conclusions from it.

In [3]:
print("Part B NPI range:", part_b.Rndrng_NPI.min(), "-", part_b.Rndrng_NPI.max())
print("Part B NPI sorted ascending:", part_b.Rndrng_NPI.is_monotonic_increasing)
print("Part B distinct providers:", part_b.Rndrng_NPI.nunique())
print("Part B distinct specialties:", part_b.Rndrng_Prvdr_Type.nunique())
print("Part B distinct states:", part_b.Rndrng_Prvdr_State_Abrvtn.nunique())
print("Part B distinct HCPCS codes:", part_b.HCPCS_Cd.nunique())
print()
print("Part D NPI range:", part_d.Prscrbr_NPI.min(), "-", part_d.Prscrbr_NPI.max())
print("Part D distinct prescribers:", part_d.Prscrbr_NPI.nunique())
print("Part D distinct specialties:", part_d.Prscrbr_Type.nunique())
print("Part D distinct states:", part_d.Prscrbr_State_Abrvtn.nunique())
print("Part D distinct generic drugs:", part_d.Gnrc_Name.nunique())

Part B NPI range: 1003000126 - 1003030651
Part B NPI sorted ascending: True
Part B distinct providers: 475
Part B distinct specialties: 63
Part B distinct states: 50
Part B distinct HCPCS codes: 939

Part D NPI range: 1003000126 - 1003013384
Part D distinct prescribers: 184
Part D distinct specialties: 40
Part D distinct states: 40
Part D distinct generic drugs: 559


## Null / suppression patterns

CMS redacts small counts to prevent patient re-identification. In Part D, `GE65_*` fields have explicit suppression flags; `Tot_Benes` (all ages) has no flag column but is blanked directly when suppressed.

In [4]:
n = len(part_d)
print(f"Part D GE65_Sprsn_Flag set: {part_d.GE65_Sprsn_Flag.notna().sum()}/{n}")
print(f"Part D GE65_Bene_Sprsn_Flag set: {part_d.GE65_Bene_Sprsn_Flag.notna().sum()}/{n}")
print(f"Part D Tot_Benes (all ages) null: {part_d.Tot_Benes.isna().sum()}/{n}")
print()
b_nulls = part_b.isna().sum()
print("Part B columns with any nulls:")
print(b_nulls[b_nulls > 0])

Part D GE65_Sprsn_Flag set: 2171/5000
Part D GE65_Bene_Sprsn_Flag set: 4335/5000
Part D Tot_Benes (all ages) null: 2577/5000

Part B columns with any nulls:
Rndrng_Prvdr_First_Name     107
Rndrng_Prvdr_MI            1672
Rndrng_Prvdr_Crdntls        175
Rndrng_Prvdr_St2           3526
Rndrng_Prvdr_RUCA             6
Rndrng_Prvdr_RUCA_Desc        6
dtype: int64


## Key findings

- **Samples are NPI-ordered, not random.** Don't treat state/specialty distributions in these samples as population-representative — a real random sample (or the full file) is needed for that.
- **Beneficiary-count suppression is common in Part D.** CMS blanks small counts to prevent re-identification: `Tot_Benes` (all ages) is null in ~52% of sample rows, `GE65_Tot_Benes` in ~87%. Needs explicit handling as a "suppressed" state, not missing-at-random, in any outlier scoring that uses these fields.
- **Part B has no such suppression** — its nulls are mundane (e.g. blank first name/MI for organizational providers, blank address line 2, a handful of unmapped RUCA codes).